In [157]:
import numpy as np
import pandas as pd
pd.set_option('display.max_rows', 24)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
import re

In [158]:
df = pd.read_csv(r"L:\emotions\nlp_eda\text.csv")
df.head()

,Unnamed: 0,text,label
0,0,i just feel really helpless and heavy hearted,4
1,1,ive enjoyed being able to slouch about relax and unwind and frankly needed it after those last few weeks around the end of uni and the expo i have lately started to find myself feeling a bit listless which is never really a good thing,0
2,2,i gave up my internship with the dmrg and am feeling distraught,4
3,3,i dont know i feel so lost,0
4,4,i am a kindergarten teacher and i am thoroughly weary of my job after having taken the university entrance exam i suffered from anxiety for weeks as i did not want to carry on with my work studies were the only alternative,4


In [159]:
df['text'].isna().sum()

0

In [160]:
df.drop("Unnamed: 0", axis=1, inplace=True)
df.head()

,text,label
0,i just feel really helpless and heavy hearted,4
1,ive enjoyed being able to slouch about relax and unwind and frankly needed it after those last few weeks around the end of uni and the expo i have lately started to find myself feeling a bit listless which is never really a good thing,0
2,i gave up my internship with the dmrg and am feeling distraught,4
3,i dont know i feel so lost,0
4,i am a kindergarten teacher and i am thoroughly weary of my job after having taken the university entrance exam i suffered from anxiety for weeks as i did not want to carry on with my work studies were the only alternative,4


In [161]:
#map labels to easy-understanding words
label_map={
    0:'sadness',
    1:'joy',
    2:'love',
    3:'anger',
    4:'fear',
    5:'surprise'
}
df['Label_String']=df['label'].map(label_map)
df.head()

,text,label,Label_String
0,i just feel really helpless and heavy hearted,4,fear
1,ive enjoyed being able to slouch about relax and unwind and frankly needed it after those last few weeks around the end of uni and the expo i have lately started to find myself feeling a bit listless which is never really a good thing,0,sadness
2,i gave up my internship with the dmrg and am feeling distraught,4,fear
3,i dont know i feel so lost,0,sadness
4,i am a kindergarten teacher and i am thoroughly weary of my job after having taken the university entrance exam i suffered from anxiety for weeks as i did not want to carry on with my work studies were the only alternative,4,fear


In [162]:
# text cleaning

# conversion to lowercase
df['text']=df['text'].str.lower()

# removal of extra whitespaces
df['text']=df['text'].str.strip()

# check for urls and image links
df[df['text'].str.contains('http')][:10]

,text,label,Label_String
36,i admit some of my inspiration to go this direction is because i am feeling a little melancholy that i won t be making my a href http www,0,sadness
180,i feel paranoid a href http cyncake,4,fear
296,i just feel like i want to change it again a href http sweet pleione,1,joy
345,i feel resistance and in this i forgive myself that i have accepted and allowed myself to not realize that i can stay awake when as i feel tired when i am resisting working on something and not give into the a href http eqafe,2,love
348,i feel so cold a href http hoyhenkeijukainen,3,anger
353,i feel bad for her but i a href http lolhunt,0,sadness
385,i feel inspired to share a class post count link href http atnelsonnest,1,joy
601,i feel needy posted by a href http jumbleupon,0,sadness
650,i have pictures everywhereee and i even managed to find places to put my paper roses my piece of scenery from senior musical the thing that says hope and the painting i made at nats which isnt that great but putting it up made me feel artistic ha a href http pics,1,joy
708,i really have to double time with all my other works so i can update this blog soon and show the world the reasons why i feel ecstatic today img src http s,1,joy


In [163]:
def get_tag_index(text:str, tag: str)->int:
    return len(text) if tag not in text else text.index(tag)


def remove_html(text: str) -> str:
    text=text.replace(" alt "," ")
    text=text.replace(" img "," ")
    
    min = get_tag_index(text, "href")
    for tag in ["a href", "www", "src", "http", "iframe", "data lang", "search form"]:
        temp = get_tag_index(text, tag)
        if temp < min:
            min = temp
    
    return text[:min]

df['text']=df['text'].apply(remove_html)

In [164]:
df[df["text"].str.contains("search form")]

,text,label,Label_String


In [165]:
def remove_xml(text: str) -> str:
    min = get_tag_index(text, "encoding utf")
    
    for tag in ["isprivate", "e mail a rel", "addthisdescription", "dhawan"]:
        temp = get_tag_index(text, tag)
        if temp < min:
            min = temp
    
    return text[:min]
        
df['text']=df['text'].apply(remove_xml)

In [166]:
def delete_before_word(text: str)-> str:
    return text[:text.find("name postacomment")].strip()[:text.find(text.split()[-1])].strip() if "postacomment" in text else text


df['text'] = df['text'].apply(delete_before_word)

In [167]:
print(df['text'].isna().sum())

0


In [168]:
df['text']=df['text'].str.strip()

In [169]:
df['text'].duplicated().sum()

23920

In [170]:
from tqdm import tqdm
def obradi_duplikate(df):
    """Ako duplikati imaju istu labelu, ostavi jedan, ako nemaju obrisi sve"""
    duplikati = df[df.duplicated(subset=['text'], keep=False)]

    for indeks, row in tqdm(duplikati.iterrows()):
        if(indeks not in duplikati.index): #zato sto ih brisem i tokom petlje
            continue

        isti_tekst_razlicite_labele = df[(df['text'] == row['text']) & (df['label'] != row['label'])]
        isti_tekst_iste_labele = df[(df['text'] == row['text']) & (df['label'] == row['label'])]
        isti_tekst_iste_labele.drop(indeks) 
        indices_to_drop = [] 

        if not isti_tekst_razlicite_labele.empty:
            indices_to_drop.extend(isti_tekst_razlicite_labele.index.tolist())
            indices_to_drop.append(indeks)
        
        if not isti_tekst_iste_labele.empty:
            indices_to_drop.extend(isti_tekst_iste_labele.index.tolist())
            indices_to_drop.remove(indeks)

        indices_to_drop = [idx for idx in indices_to_drop if idx in df.index]
        df = df.drop(indices_to_drop)
        duplikati=duplikati.drop(indices_to_drop)

    return df

In [171]:
contractions = {
"ain't": "are not",
"aren't": "are not",
"can't": "cannot",
"can't've": "cannot have",
"'cause": "because",
"could've": "could have",
"couldn't": "could not",
"couldn't've": "could not have",
"didn't": "did not",
"doesn't": "does not",
"don't": "do not",
"hadn't": "had not",
"hadn't've": "had not have",
"hasn't": "has not",
"haven't": "have not",
"he'd": "he had",
"he'd've": "he would have",
"he'll": "he will",
"he'll've": "he will have",
"he's": "he is",
"how'd": "how did",
"how'd'y": "how do you",
"how'll": "how will",
"how's": "how is",
"i'd": "I had",
"i'd've": "I would have",
"i'll": "I will",
"i'll've": "I will have",
"i'm": "I am",
"i've": "I have",
"isn't": "is not",
"it'd": "it had / it would",
"it'd've": "it would have",
"it'll": "it will",
"it'll've": "it will have",
"it's": "it is",
"let's": "let us",
"ma'am": "madam",
"mayn't": "may not",
"might've": "might have",
"mightn't": "might not",
"mightn't've": "might not have",
"must've": "must have",
"mustn't": "must not",
"mustn't've": "must not have",
"needn't": "need not",
"needn't've": "need not have",
"o'clock": "of the clock",
"oughtn't": "ought not",
"oughtn't've": "ought not have",
"shan't": "shall not",
"sha'n't": "shall not",
"shan't've": "shall not have",
"she'd": "she would",
"she'd've": "she would have",
"she'll": "she will",
"she'll've": "she will have",
"she's": "she is",
"should've": "should have",
"shouldn't": "should not",
"shouldn't've": "should not have",
"so've": "so have",
"so's": "so is",
"that'd": "that would",
"that'd've": "that would have",
"that's": "that is",
"there'd": "there would",
"there'd've": "there would have",
"there's": "there is",
"they'd": "they would",
"they'd've": "they would have",
"they'll": "they will",
"they'll've": "they will have",
"they're": "they are",
"they've": "they have",
"to've": "to have",
"wasn't": "was not",
"we'd": "we would",
"we'd've": "we would have",
"we'll": "we will",
"we'll've": "we will have",
"we're": "we are",
"we've": "we have",
"weren't": "were not",
"what'll": "what will",
"what'll've": "what will have",
"what're": "what are",
"what's": "what is",
"what've": "what have",
"when's": "when is",
"when've": "when have",
"where'd": "where did",
"where's": "where is",
"where've": "where have",
"who'll": "who will",
"who'll've": "who will have",
"who's": "who is",
"who've": "who have",
"why's": "why is",
"why've": "why have",
"will've": "will have",
"won't": "will not",
"won't've": "will not have",
"would've": "would have",
"wouldn't": "would not",
"wouldn't've": "would not have",
"y'all": "you all",
"y'all'd": "you all would",
"y'all'd've": "you all would have",
"y'all're": "you all are",
"y'all've": "you all have",
"you'd": "you would",
"you'd've": "you would have",
"you'll": "you will",
"you'll've": "you will have",
"you're": "you are",
"you've": "you have"
}

In [172]:
#same as previous without apostrophes in keys
contractions_wo_punct = {key.replace("'", ""): value for key, value in contractions.items()}

In [173]:
contractions_wo_punct

{'aint': 'are not',
 'arent': 'are not',
 'cant': 'cannot',
 'cantve': 'cannot have',
 'cause': 'because',
 'couldve': 'could have',
 'couldnt': 'could not',
 'couldntve': 'could not have',
 'didnt': 'did not',
 'doesnt': 'does not',
 'dont': 'do not',
 'hadnt': 'had not',
 'hadntve': 'had not have',
 'hasnt': 'has not',
 'havent': 'have not',
 'hed': 'he had',
 'hedve': 'he would have',
 'hell': 'he will',
 'hellve': 'he will have',
 'hes': 'he is',
 'howd': 'how did',
 'howdy': 'how do you',
 'howll': 'how will',
 'hows': 'how is',
 'id': 'I had',
 'idve': 'I would have',
 'ill': 'I will',
 'illve': 'I will have',
 'im': 'I am',
 'ive': 'I have',
 'isnt': 'is not',
 'itd': 'it had / it would',
 'itdve': 'it would have',
 'itll': 'it will',
 'itllve': 'it will have',
 'its': 'it is',
 'lets': 'let us',
 'maam': 'madam',
 'maynt': 'may not',
 'mightve': 'might have',
 'mightnt': 'might not',
 'mightntve': 'might not have',
 'mustve': 'must have',
 'mustnt': 'must not',
 'mustntve': 'mu

In [174]:
def replace_contractions1(text):
    words=text.split()
    for i, word in enumerate(words):
        if word.lower() in contractions:
            words[i]=contractions[word.lower()].lower()
    return ' '.join(words)

def replace_contractions2(text):
    words=text.split()
    for i, word in enumerate(words):
        if word.lower() in contractions_wo_punct:
            words[i]=contractions_wo_punct[word.lower()].lower()
    return ' '.join(words)
df['text']=df['text'].apply(replace_contractions1).apply(replace_contractions2)

In [175]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
stop_words=set(stopwords.words('english'))

def remove_stopwords(text):
    words=text.split()
    filtered_words=[word for word in words if word.lower() not in stop_words]
    return ' '.join(filtered_words)

df['text']=df['text'].apply(remove_stopwords)
df.head()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\lukic\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,text,label,Label_String
0,feel really helpless heavy hearted,4,fear
1,enjoyed able slouch relax unwind frankly needed last weeks around end uni expo lately started find feeling bit listless never really good thing,0,sadness
2,gave internship dmrg feeling distraught,4,fear
3,know feel lost,0,sadness
4,kindergarten teacher thoroughly weary job taken university entrance exam suffered anxiety weeks want carry work studies alternative,4,fear


In [176]:
chatwrds = {
    "LOL": "Laughing out loud",
    "ASAP": "As soon as possible",
    "FYI": "For your information",
    "G2G": "Got to go",
    "FB": "Facebook",
    "MSG": "Message",
    "TTYL": "Talk to you later",
    "IMO": "In my opinion",
    "PAW": "Parents are watching",
    "PITR": "Parent in the room",
    "PBB": "Parent behind back",
    "POMS": "Parent over my shoulder",
    "KPC": "Keeping parents clueless",
    "PAH": "Parent at home",
    "HIFW": "How I feel when",
    "TFW": "That feeling when",
    "MFW": "My face when",
    "MRW": "My reaction when",
    "IFYP": "I feel your pain",
    "TNTL": "Trying not to laugh",
    "JK": "Just kidding",
    "IDC": "I don't care",
    "ILY": "I love you",
    "IMU": "I miss you",
    "ADIH": "Another day in hell",
    "ZZZ": "Sleeping bored tired",
    "WYWH": "Wish you were here",
    "BAE": "Before anyone else",
    "FIMH": "Forever in my heart",
    "BSAAW": "Big smile and a wink",
    "BWL": "Bursting with laughter",
    "LMAO": "Laughing my ass off",
    "BFF": "Best friends forever",
    "BESTIE": "Best friend",
    "BESTIES": "Best friends",
    "CSL": "Can't stop laughing",
    "IMOIMHO": "In my opinion In my humble opinion",
    "OMDB": "Over my dead body",
    "NTH": "Nice to have",
    "POV": "Point of View",
    "TBH": "To be honest",
    "FTW": "For the win",
    "WTF": "What the fuck",
    "FTL": "For the loss",
    "YNK": "You never know",
    "SMH": "Shaking my head",
    "SRSLY": "Seriously",
    "YGTR": "You got that right",
    "GMTA": "Great minds think alike",
    "AYMM": "Are you my mother",
    "CWOT": "Complete waste of time",
    "IANAL": "I am not a lawyer",
    "AFAIK": "As far as I know",
    "AFAIR": "As far as I remember",
    "AFAIC": "As far as I'm concerned",
    "ASL": "Age sex location",
    "AAMOF": "As a matter of fact",
    "FWIW": "For what it's worth",
    "YMMV": "Your mileage may vary",
    "IIRC": "If I remember correctly",
    "DM": "Direct message",
    "AFAICT": "As far as I can tell",
    "TLDR": "Too long didn't read",
    "IRL": "In real life",
    "TIL": "Today I learned",
    "SOML": "Story of my life",
    "EMBM": "Early morning business meeting",
    "J4F": "Just for fun",
    "JSYK": "Just so you know",
    "FAWC": "For anyone who cares",
    "RLRT": "Real life retweet",
    "OH": "Overheard",
    "WUZUP": "What's up",
    "CS": "Career suicide",
    "DWH": "During work hours",
    "OMW": "On my way",
    "GRATZ": "Congratulations",
    "GL": "Good luck",
    "IDK": "I don't know",
    "BRB": "Be right back",
    "W8": "Wait",
    "NFS": "Not for sale",
    "B4N": "Bye for now",
    "B@U": "Back at you",
    "BBBG": "Bye bye be good",
    "BBIAS": "Be back in a sec",
    "RUOK": "Are you OK",
    "CYT": "See you tomorrow",
    "DBMIB": "Don't bother me I'm busy",
    "GFN": "Gone for now",
    "AFK": "Away from keyboard",
    "NSFW": "Not safe for work",
    "NSFL": "Not safe for life",
    "SFW": "Safe for work",
    "GRAS": "Generally recognised as safe",
    "NBD": "Not big deal",
    "4AO": "For adults only",
    "OP": "Original poster",
    "PPL": "People",
    "PPLE": "People",
    "ICYMI": "In case you missed it",
    "JIC": "Just in case",
    "NAGI": "Not a good idea",
    "GOI": "Get over it",
    "RBTL": "Read between the lines",
    "AYOR": "At your own risk",
    "DIY": "Do it yourself",
    "E123": "Easy as one two three",
    "GAHOY": "Get a hold of yourself",
    "TMB": "Tweet me back",
    "WTPA": "Where the party at",
    "DAE": "Does anyone else",
    "PRT": "Please retweet",
    "PTB": "Please text back",
    "TIA": "Thanks in advance",
    "BUMP": "Bring up my post",
    "PLS": "Please",
    "TMRW": "Tomorrow",
    "GUNNA":"Going to",
    "GONNA": "Going to",
    "YKNOW": "You know",
    "YK": "You know",
    "CONVO": "Conversation",
    "BDAY": "Birthday",
    "LYK": "Let you know",
    "BOYF": "Boyfriend",
    "BF": "Boyfriend",
    "GIRLF": "Girlfriend",
    "GF": "Girlfriend",
    "BECOZ": "Because",
    "BECOS":"Because",
    "BCOZ": "Because",
    "BCOS":"Because",
    "BCUZ": "Because",
    "BCUS": "Because",
    "BC": "Because",
    "BCAUSE":"Because",
    "BCAUS": "Because",
    "BCZ": "Because",
    "NVM": "Never mind",
    "CMON": "Come on",
    "OMG": "Oh my God",
    "OMFG": "Oh my fucking God",
    "ROFL": "Rolling on the floor laughing",
    "GTFO": "Get the fuck out",
    "THANX": "Thank you",
    "THANKS": "Thanks",
    "OBVS": "Obviously",
    "GUAC": "Guacamole",
    "IMHO": "In my humble opinion",
    "IDKY": "I don't know why",
    "RMBR": "Remember",
    "FCKN": "Fucking",
    "DAFUQ": "What the fuck",
    "FRNDS": "Friends",
    "MWAH": "Kiss",
    "FKING": "Fucking",
    "ANONS": "Anounymous",
    "RILLY": "Really",
    "RLLY": "Really",
    "BTWN": "Between",
    "CYA": "See you",
    "CIYA": "See you",
    "DAYUM": "Damn",
    "ATMO": "At the moment",
    "HMWK": "Homework",
    "WYA": "Where are you at",
    "FAV": "Favorite",
    "PERV": "Pervert",
    "STH": "Something",
    "SMTH": "Something",
    "SOMETH": "Something",
    "IDEK": "I don't even know",
    "FKERS": "Fuckers",
    "FKED": "Fucked",
    "YAY": "Celebration",
    "XD": "Laughter",
    "BLA": "Tedious",
    "BLAH": "Tedious",
    "YEA": "Yes",
    "YEAH": "Yes",
    "GRR": "Snarl",
    "SHH": "Hush",
    "DEFO": "Definitely",
    "TYS": "Thank you so much",
    "TYSM": "Thank you so much",
    "TXT": "Text",
    "BTW": "By the way",
    "FAV": "Favorite"
}


In [177]:
#same as previous without apostrophes in values
chatwrds_wo_punct = {key: value.replace("'", "") for key, value in chatwrds.items()}

In [178]:
chatwrds_wo_punct

{'LOL': 'Laughing out loud',
 'ASAP': 'As soon as possible',
 'FYI': 'For your information',
 'G2G': 'Got to go',
 'FB': 'Facebook',
 'MSG': 'Message',
 'TTYL': 'Talk to you later',
 'IMO': 'In my opinion',
 'PAW': 'Parents are watching',
 'PITR': 'Parent in the room',
 'PBB': 'Parent behind back',
 'POMS': 'Parent over my shoulder',
 'KPC': 'Keeping parents clueless',
 'PAH': 'Parent at home',
 'HIFW': 'How I feel when',
 'TFW': 'That feeling when',
 'MFW': 'My face when',
 'MRW': 'My reaction when',
 'IFYP': 'I feel your pain',
 'TNTL': 'Trying not to laugh',
 'JK': 'Just kidding',
 'IDC': 'I dont care',
 'ILY': 'I love you',
 'IMU': 'I miss you',
 'ADIH': 'Another day in hell',
 'ZZZ': 'Sleeping bored tired',
 'WYWH': 'Wish you were here',
 'BAE': 'Before anyone else',
 'FIMH': 'Forever in my heart',
 'BSAAW': 'Big smile and a wink',
 'BWL': 'Bursting with laughter',
 'LMAO': 'Laughing my ass off',
 'BFF': 'Best friends forever',
 'BESTIE': 'Best friend',
 'BESTIES': 'Best friends',

In [179]:
#switching chat words, uses words without apostrophes
def replace_chat_words(text):
    words=text.split()
    for i, word in enumerate(words):
        if word.upper() in chatwrds_wo_punct:
            words[i]=chatwrds_wo_punct[word.upper()].lower()
    return ' '.join(words)

df['text']=df['text'].apply(replace_chat_words)

In [180]:
df['text']=df['text'].str.lower() #bc chat words

In [181]:
#removing interpunctions
import string
PUNCT_TO_REMOVE = string.punctuation
def remove_punctuation(text):
    """custom function to remove the punctuation"""
    return text.translate(str.maketrans('', '', PUNCT_TO_REMOVE))

df["text"] = df["text"].apply(remove_punctuation)
df.head()

,text,label,Label_String
0,feel really helpless heavy hearted,4,fear
1,enjoyed able slouch relax unwind frankly needed last weeks around end uni expo lately started find feeling bit listless never really good thing,0,sadness
2,gave internship dmrg feeling distraught,4,fear
3,know feel lost,0,sadness
4,kindergarten teacher thoroughly weary job taken university entrance exam suffered anxiety weeks want carry work studies alternative,4,fear


In [182]:
# changing hahahehehihi to laughter
pattern = r'(ha|he|hi|ah|eh|ih){2,}'
fw_l=[]
def replace_laughter(word):
    if re.search(pattern, word):
        fw_l.append(word)
        return 'laughter'
    else:
        return word
    
df['text']=df['text'].apply(lambda x: ' '.join([replace_laughter(word) for word in x.split()]))

In [183]:
df[df['text'].str.contains("lolita")]

,text,label,Label_String
8137,wish told beforehand would dressed little feel somewhat place casual ita lolita skirt navy hoody laughter still glad taken,1,joy
12016,actually put writing long feel casual lolita definitely misnomer generally often misunderstood branch lolita,1,joy
24906,think telling parents laughs really like gothic lolita visual kei feel regretful could go jrock revolution cares ago,0,sadness
53574,look cute lolita also feel glamorous sexy,1,joy
56597,always feel cute pretty wearing lolita,1,joy
68485,feel dirty listening melodic synth lolita voices,0,sadness
107614,selling feel simply pretty feel elegant wear everyday casual lolita,1,joy
114172,think lolita blog feeling kind unloved moment still one default blogger layouts,0,sadness
117605,dip saturated literary experience feeling tank little low jews god history parents insist must read thereby rendering attitude toward cautious actively reading nabokovs lolita lauren belfers fierce radiance,0,sadness
129455,feel extremely honored chosen lolita long also really expect anything sent application,1,joy


In [184]:
fw_l

['haha',
 'haha',
 'haha',
 'haha',
 'hahaha',
 'hehe',
 'hahaha',
 'hehehe',
 'haha',
 'hihi',
 'hehe',
 'haha',
 'hahaha',
 'haha',
 'hahahaha',
 'hehe',
 'haha',
 'hahaha',
 'haha',
 'hahaha',
 'shihab',
 'hehe',
 'hahaha',
 'haha',
 'hahaha',
 'hahaha',
 'hehe',
 'hahaha',
 'hahah',
 'hahaha',
 'hehe',
 'ahahahah',
 'hahahaha',
 'haha',
 'haha',
 'ahahah',
 'hehe',
 'haha',
 'hahah',
 'haha',
 'hehe',
 'haha',
 'hahaha',
 'haha',
 'haha',
 'haha',
 'haha',
 'hahahah',
 'haha',
 'haha',
 'haha',
 'haha',
 'hahaha',
 'ahaha',
 'haha',
 'hahaha',
 'haha',
 'haha',
 'hahaha',
 'hahaha',
 'haha',
 'panghihinayang',
 'ahhahahahahhaaa',
 'haha',
 'haha',
 'haha',
 'haha',
 'haha',
 'haha',
 'haha',
 'hahaha',
 'hehe',
 'haha',
 'hahah',
 'haha',
 'haha',
 'haha',
 'haha',
 'haha',
 'hehehe',
 'haha',
 'haha',
 'haha',
 'hehe',
 'hahaha',
 'hahah',
 'haha',
 'hehe',
 'haha',
 'haha',
 'hahaha',
 'haha',
 'haha',
 'hahahaha',
 'hahah',
 'haha',
 'hahha',
 'haha',
 'haha',
 'haha',
 'haha',


In [185]:
#switching lol looool
pattern = r'^[lo]*lol[lo]*$'
fw_lol=[] #words that satisfy regex
def replace_lol(word):
    word_changed = re.sub(r'lo+l', 'lol', word) #if it finds looooll first cnanges it to loll
    if re.search(pattern, word_changed):
        fw_lol.append(word)
        return 'laughter'
    else:
        return word
    
df['text']=df['text'].apply(lambda x: ' '.join([replace_lol(word) for word in x.split()]))

In [186]:
fw_lol

['lolll',
 'loool',
 'lolol',
 'lololol',
 'lool',
 'lool',
 'lolol',
 'lolol',
 'lolol',
 'lolol',
 'lolo',
 'lololol',
 'lolololololol',
 'lolol',
 'lolllll',
 'loooooool',
 'lolllol',
 'lolol',
 'lololol',
 'lololol',
 'lolol']

In [187]:
#switching xd xdd xxd
pattern = r'^x+d+'
fw_xd=[] #words that satisfy regex
def replace_xd(word):
    if re.search(pattern, word):
        fw_xd.append(word)
        return 'laughter'
    else:
        return word
    
df['text']=df['text'].apply(lambda x: ' '.join([replace_xd(word) for word in x.split()]))

In [188]:
fw_xd

['xddd',
 'xdd',
 'xddddd',
 'xdd',
 'xdd',
 'xdd',
 'xdd',
 'xdddd',
 'xdddddd',
 'xdd',
 'xdd',
 'xdddddd',
 'xdd']

In [189]:
#changing x xx xxx to kiss
pattern = r'^x+$'
fw_x=[] #words that satisfy regex
def replace_x(word):
    if re.search(pattern, word):
        fw_x.append(word)
        return 'kiss'
    else:
        return word
    
df['text']=df['text'].apply(lambda x: ' '.join([replace_x(word) for word in x.split()]))

In [190]:
fw_x 

['xx',
 'x',
 'x',
 'xx',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'xxx',
 'xxxxxxx',
 'x',
 'x',
 'x',
 'x',
 'xxxxx',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'xx',
 'xxx',
 'x',
 'x',
 'xx',
 'x',
 'x',
 'x',
 'x',
 'xx',
 'x',
 'xx',
 'xxx',
 'x',
 'x',
 'xxxxxx',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'xxx',
 'xxx',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'xx',
 'xx',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'xxx',
 'xxxxxxxxxxx',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'xx',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'xxx',
 'xxxx',
 'x',
 'xxx',
 'x',
 'x',
 'xxx',
 'x',
 'xxxx',
 'xx',
 'x',
 'xx',
 'x',
 'x',
 'x',
 'xxx',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 'x',
 '

In [191]:
#changing xoxo to hugs and kisses
pattern = r'^(?=.*x)(?=.*o)[xo]+$'
fw_xo=[] #words that satisfy regex
def replace_xo(word):
    if re.search(pattern, word):
        fw_xo.append(word)
        return 'hugs and kisses'
    else:
        return word
    
df['text']=df['text'].apply(lambda x: ' '.join([replace_xo(word) for word in x.split()]))

In [192]:
fw_xo

['xox',
 'xoxo',
 'xo',
 'xoxo',
 'xoxoxoxoxoxoxoxoxoxoxoxoxoxoxoxoxoxoxoxoxoxoxo',
 'xo',
 'xo',
 'xo',
 'ox',
 'xoxo',
 'xoxo',
 'ox',
 'xo',
 'xoxo',
 'xoxoxo',
 'xoxoxo',
 'xoxo',
 'xoxoxo',
 'xxxxoooo',
 'xoxoxo',
 'xoxo',
 'xoxo',
 'xo',
 'xoxo',
 'xoxo',
 'ox',
 'xxo',
 'ox',
 'ox',
 'ox']

In [193]:
#swtiching omgs
pattern = r'^o+m+g+$'
fw_omg=[] #words that satisfy regex
def replace_omg(word):
    if re.search(pattern, word):
        fw_omg.append(word)
        return 'oh my god'
    else:
        return word
    
df['text']=df['text'].apply(lambda x: ' '.join([replace_omg(word) for word in x.split()]))

In [194]:
fw_omg 

['oooommmggggggg', 'ooommmmggggg']

In [195]:
#switching zzzzz
pattern = r'^z{2,}s*$'
fw_zz=[] #words that satisfy regex
def replace_zz(word):
    if re.search(pattern, word):
        fw_zz.append(word)
        return 'sleeping bored tired'
    else:
        return word
    
df['text']=df['text'].apply(lambda x: ' '.join([replace_zz(word) for word in x.split()]))

In [196]:
fw_zz

['zz', 'zz', 'zzzzzzs', 'zzzzzz', 'zzzz', 'zzzzzzz']

In [197]:
#switching uh ah oh eh
pattern = r'\b(?:o+h+|u+h+|e+h+|a+h+)\b'
fw_uh=[] #words that satisfy regex
def replace_oh(word):
    if re.search(pattern, word):
        fw_uh.append(word)
        return 'sigh'
    else:
        return word
    
df['text']=df['text'].apply(lambda x: ' '.join([replace_oh(word) for word in x.split()]))

In [198]:
fw_uh 

['oh',
 'ah',
 'ohhh',
 'uhhhh',
 'ah',
 'ooh',
 'eh',
 'aaaah',
 'ah',
 'oh',
 'ah',
 'oh',
 'ahh',
 'ah',
 'uhhh',
 'uh',
 'uhh',
 'ooh',
 'ahhhh',
 'eh',
 'ahhh',
 'ahhhhh',
 'eh',
 'eh',
 'ah',
 'oh',
 'oooh',
 'eh',
 'uh',
 'ohhh',
 'ahhh',
 'ooooh',
 'oohhh',
 'oohhhhh',
 'uhh',
 'ah',
 'ooohhh',
 'ah',
 'ah',
 'ooh',
 'ooh',
 'eh',
 'oh',
 'ooh',
 'eh',
 'ohhhh',
 'uh',
 'uuuhhh',
 'ahhh',
 'ah',
 'ah',
 'eh',
 'oh',
 'uh',
 'uhh',
 'oh',
 'ah',
 'ohh',
 'aahhh',
 'eh',
 'oh',
 'oh',
 'ohhhhh',
 'ohh',
 'ohhh',
 'ah',
 'ahh',
 'eh',
 'ooooh',
 'oh',
 'oh',
 'oh',
 'uh',
 'ah',
 'oh',
 'oooh',
 'oh',
 'ooh',
 'ah',
 'oooh',
 'uh',
 'ahh',
 'eh',
 'uh',
 'uhh',
 'ah',
 'oh',
 'oooh',
 'ooh',
 'ah',
 'oh',
 'oh',
 'eh',
 'uhhhh',
 'ahhh',
 'ah',
 'eh',
 'ooh',
 'oh',
 'eh',
 'ooh',
 'ooooh',
 'ah',
 'eh',
 'aaah',
 'eh',
 'ah',
 'ahh',
 'ahhhhhhhhhhhhhhhhhhhhhhhh',
 'oh',
 'ah',
 'ah',
 'ahhhhh',
 'uh',
 'uh',
 'oh',
 'oh',
 'ah',
 'ah',
 'aahhh',
 'ooh',
 'ah',
 'oh',
 'oh',
 'ah'

In [199]:
def remove_misc(text: str) -> str:
    min = get_tag_index(text, "onclick")
    
    for tag in ["class alignleft", "class alignright", "class post", "class rsswidget", "instaquote", "span class", "div content", "right post", "div content", "article class", "source rel", "target blank", " rel ", "mp link", "pagetitle", "onblur", "border id", "oncontextmenu", "onmouseover", "pagetype", "class aligncenter", "size full wp", "width height"]:
        temp = get_tag_index(text, tag)
        if temp < min:
            min = temp
    
    return text[:min]
        
df.drop(df[df["text"].str.contains("himchan")].index, inplace=True) #not english
df['text']=df['text'].apply(remove_misc)

In [201]:
from itertools import groupby

def remove_three_peat(text: str)->str:
    for _, group in groupby(text):
        if len(list(group)) >= 3:
            text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    return text

df['text'] = df['text'].apply(remove_three_peat)

df=obradi_duplikate(df)
df['text'].duplicated().sum()

65684it [31:37, 34.62it/s]  


0

In [202]:
df.shape

(356403, 3)

In [204]:
df2 = df.drop("Label_String", axis=1)
df2.head()

,text,label
1,enjoyed able slouch relax unwind frankly needed last weeks around end uni expo lately started find feeling bit listless never really good thing,0
2,gave internship dmrg feeling distraught,4
3,know feel lost,0
4,kindergarten teacher thoroughly weary job taken university entrance exam suffered anxiety weeks want carry work studies alternative,4
5,beginning feel quite disheartened,0


In [220]:
df.to_csv("modelling_after_pp_V1.csv")